# 实验：vLLM + Agent 天气工具调用

> 这份 notebook 用一个完全本地的模拟天气工具，测试 `vLLM` 后端和 `agent/` 侧的 OpenAI-compatible tool calling 是否打通。

> 不依赖检索语料，也不依赖外部天气 API。


## 0. 环境与服务说明

假设你已经在服务器上启动好了 vLLM，并为当前模型配置了可用的 tool parser。

这个 notebook 的目标很简单：

1. 给模型一个天气问题
2. 让模型调用 `get_weather`
3. 本地执行模拟工具
4. 把工具结果回填给模型，观察最终回答


In [ ]:
!python --version
!pip install -r agent/requirements.txt


## 1. 初始化 vLLM 客户端

这里的 `MODEL_NAME` 可以切换成你当前起好的模型名，比如：

- `qwen_auto`（推荐）
- `pangu_auto`

In [ ]:
from pathlib import Path
import json
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agent.vllm_client import VLLMClient

VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'
#MODEL_NAME = 'pangu_auto'
MODEL_NAME = 'qwen_auto'
API_KEY = 'dummy'

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)

## 2. 定义一个完全本地的模拟天气工具

为了让测试稳定，我们把天气数据写死在 notebook 里。


In [22]:
MOCK_WEATHER_DB = {
    '南京': {'temperature_c': 24, 'condition': '多云', 'humidity': 58, 'wind': '东北风 3级'},
    '上海': {'temperature_c': 27, 'condition': '晴', 'humidity': 67, 'wind': '东南风 4级'},
    '北京': {'temperature_c': 21, 'condition': '晴转阴', 'humidity': 42, 'wind': '西北风 2级'},
    '杭州': {'temperature_c': 25, 'condition': '小雨', 'humidity': 81, 'wind': '东风 2级'},
}


def get_weather(city: str):
    city = city.strip()
    if city not in MOCK_WEATHER_DB:
        return {
            'city': city,
            'found': False,
            'message': 'mock weather data not found',
        }
    return {
        'city': city,
        'found': True,
        **MOCK_WEATHER_DB[city],
    }


tool_specs = [
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': 'Get mock weather data for a given city.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'city': {
                        'type': 'string',
                        'description': 'City name in Chinese, such as 南京 or 上海.',
                    }
                },
                'required': ['city'],
            },
        },
    }
]

tool_registry = {
    'get_weather': get_weather,
}

print(json.dumps(tool_specs, ensure_ascii=False, indent=2))


[
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get mock weather data for a given city.",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "City name in Chinese, such as 南京 or 上海."
          }
        },
        "required": [
          "city"
        ]
      }
    }
  }
]


## 3. 工具执行辅助函数


In [23]:
def execute_tool_call(tool_call, registry):
    function = tool_call.get('function', {})
    name = function.get('name', '')
    arguments = function.get('arguments', '{}')
    if isinstance(arguments, str):
        arguments = json.loads(arguments)
    result = registry[name](**arguments)
    return {
        'tool_name': name,
        'arguments': arguments,
        'tool_result': result,
    }


## 4. 最小 agent loop

这里故意把系统提示写得很明确：只要用户问天气，必须先调用 `get_weather`，不能靠常识直接回答。


In [24]:
def run_weather_agent(query: str, max_rounds: int = 4, max_tokens: int = 512):
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a weather assistant. '
                'For any weather-related question, you must call the get_weather tool before answering. '
                'Do not answer weather questions from memory. '
                'After receiving tool results, answer in Chinese and keep the answer concise.'
            ),
        },
        {'role': 'user', 'content': query},
    ]
    trajectory = []

    for round_id in range(1, max_rounds + 1):
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.0,
            max_tokens=max_tokens,
            tools=tool_specs,
            tool_choice='auto',
        )
        message = response['choices'][0]['message']
        raw_content = message.get('content', '')
        tool_calls = message.get('tool_calls') or []

        step = {
            'round_id': round_id,
            'assistant_content': raw_content,
            'tool_calls': tool_calls,
            'usage': response.get('usage', {}),
        }
        trajectory.append(step)

        assistant_message = {'role': 'assistant', 'content': raw_content}
        if tool_calls:
            assistant_message['tool_calls'] = tool_calls
        messages.append(assistant_message)

        if not tool_calls:
            return {
                'query': query,
                'status': 'completed',
                'final_output': raw_content,
                'trajectory': trajectory,
                'messages': messages,
            }

        tool_results = []
        for tool_call in tool_calls:
            executed = execute_tool_call(tool_call, tool_registry)
            tool_results.append(executed)
            messages.append(
                {
                    'role': 'tool',
                    'tool_call_id': tool_call['id'],
                    'content': json.dumps(executed['tool_result'], ensure_ascii=False),
                }
            )
        step['tool_results'] = tool_results

    return {
        'query': query,
        'status': 'max_rounds_reached',
        'final_output': '',
        'trajectory': trajectory,
        'messages': messages,
    }


## 5. 单条测试

这个例子通常会触发两次工具调用：先查南京，再查上海。


In [25]:
demo_query = '请比较南京和上海今天的天气，告诉我哪个更适合穿短袖，并简要说明原因。'
demo = run_weather_agent(demo_query, max_rounds=4)

print('status:', demo['status'])
print('\nfinal output:\n')
print(demo['final_output'])
print('\ntrajectory preview:\n')
print(json.dumps(demo['trajectory'], ensure_ascii=False, indent=2)[:6000])


status: completed

final output:

 [unused16] 根据获取的天气信息：

**南京今天天气：**
- 温度：24°C
- 天气状况：多云
- 湿度：58%
- 风力：东北风 3级

**上海今天天气：**
- 温度：27°C
- 天气状况：晴
- 湿度：67%
- 风力：东南风 4级

**分析：**
- 上海温度更高（27°C vs 24°C）
- 上海湿度更高（67% vs 58%）
- 风力上海略强（4级 vs 3级）

**结论：**
上海今天天气更热，湿度更高，风力略强。因此，**南京更适合穿短袖**，因为温度相对较低，体感更为舒适。

trajectory preview:

[
  {
    "round_id": 1,
    "assistant_message": {
      "role": "assistant",
      "reasoning_content": null,
      "content": "让我先获取南京和上海今天的天气信息：\n\n**调用工具：**\n\n\n**调用工具：**",
      "tool_calls": [
        {
          "id": "chatcmpl-tool-ae84385bd4a349c99d419a76f6898429",
          "type": "function",
          "function": {
            "name": "get_weather",
            "arguments": "{\"city\": \"南京\"}"
          }
        },
        {
          "id": "chatcmpl-tool-b3fa56976a1d48948e0e1b9ec6f22169",
          "type": "function",
          "function": {
            "name": "get_weather",
            "arguments": "{\"city\": \"上海\"}"
          }
        }
      ]
    }

## 6. 再试一个单城市问题


In [ ]:
single_city_demo = run_weather_agent('南京今天什么天气？', max_rounds=4)
print(single_city_demo['final_output'])
